In [0]:
%pip install gspread google-auth pandas
dbutils.library.restartPython()

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict, 
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1dsomIdRf4vlTAfoMQ-rIq1brARZmUUVTjUEmXU51BBk"
sh = gc.open_by_key(SHEET_ID)

# Ensure the "Data" tab exists and has headers if it's brand new
try:
    ws = sh.worksheet("Data")
except gspread.exceptions.WorksheetNotFound:
    ws = sh.add_worksheet(title="Data", rows=1000, cols=15)
    ws.append_row(['Date', 'Hour', 'Vol_OSR_PiE', 'Vol_OSR_Topup', 'Vol_E3_Packing', 
                   'Vol_Parcel_Sortation', 'Vol_Parcel_Induct', 'Vol_Inbound_Decanting', 
                   'Vol_OSR_Decanting', 'Vol_BCR_Inducting', 'Vol_E1_E2_Inducting'])

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))

df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

In [0]:
hourly_volume_query = """
WITH base AS (
    SELECT
        date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'yyyy-MM-dd') AS Date,
        hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London')) AS Hour,
        PAYLOAD_QUANTITY AS qty,
        CASE 
            WHEN PAYLOAD_AREACODE IN ('Pick','Pie Station') AND PAYLOAD_EVENTTYPE IN ('MSKU', 'PSKU') THEN 'OSR PiE'
            WHEN PAYLOAD_AREACODE IN ('Topup','TopUp','PiOrQi') AND PAYLOAD_EVENTTYPE = 'TPUT' THEN 'OSR Topup'
            WHEN PAYLOAD_AREACODE = 'E3 - Packing' AND PAYLOAD_EVENTTYPE = 'PackingItemScannedEvent' THEN 'E3 Packing'
            WHEN PAYLOAD_EVENTTYPE = 'ParcelSortedToSack' THEN 'Parcel Sortation'
            WHEN PAYLOAD_AREACODE = 'Parcel Induct' AND PAYLOAD_EVENTTYPE = 'SPAR' THEN 'Parcel Induct'
            WHEN PAYLOAD_AREACODE = 'Inbound Decanting' AND PAYLOAD_EVENTTYPE = 'DECN' THEN 'Inbound Decanting'
            WHEN PAYLOAD_AREACODE = 'OSR Decanting' AND PAYLOAD_EVENTTYPE = 'ODEC' THEN 'OSR Decanting'
            WHEN PAYLOAD_AREACODE = 'Induct from E1/E2' AND PAYLOAD_EVENTTYPE = 'SPOS' AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%') THEN 'BCR Inducting'
            WHEN PAYLOAD_AREACODE = 'Induct from E1/E2' AND PAYLOAD_EVENTTYPE = 'SPOS' AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%PIE4EDW%') THEN 'E1/E2 Inducting'
        END AS work_area
    FROM landing_bonus_hub_event_parsed
    WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'X'
      -- Dynamically filter for the exact previous hour in local UK time
      AND hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP), 'Europe/London')) = hour(from_utc_timestamp(current_timestamp() - INTERVAL 1 HOUR, 'Europe/London'))
      AND to_date(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP), 'Europe/London')) = to_date(from_utc_timestamp(current_timestamp() - INTERVAL 1 HOUR, 'Europe/London'))
)
SELECT 
    Date,
    Hour,
    SUM(CASE WHEN work_area = 'OSR PiE' THEN qty ELSE 0 END) AS Vol_OSR_PiE,
    SUM(CASE WHEN work_area = 'OSR Topup' THEN qty ELSE 0 END) AS Vol_OSR_Topup,
    SUM(CASE WHEN work_area = 'E3 Packing' THEN qty ELSE 0 END) AS Vol_E3_Packing,
    SUM(CASE WHEN work_area = 'Parcel Sortation' THEN qty ELSE 0 END) AS Vol_Parcel_Sortation,
    SUM(CASE WHEN work_area = 'Parcel Induct' THEN qty ELSE 0 END) AS Vol_Parcel_Induct,
    SUM(CASE WHEN work_area = 'Inbound Decanting' THEN qty ELSE 0 END) AS Vol_Inbound_Decanting,
    SUM(CASE WHEN work_area = 'OSR Decanting' THEN qty ELSE 0 END) AS Vol_OSR_Decanting,
    SUM(CASE WHEN work_area = 'BCR Inducting' THEN qty ELSE 0 END) AS Vol_BCR_Inducting,
    SUM(CASE WHEN work_area = 'E1/E2 Inducting' THEN qty ELSE 0 END) AS Vol_E1_E2_Inducting
FROM base
WHERE work_area IS NOT NULL
GROUP BY Date, Hour
"""

# Execute query and convert to Pandas
hourly_volumes_df = spark.sql(hourly_volume_query)
pdf = hourly_volumes_df.toPandas().fillna(0)

# Append to the Google Sheet
if not pdf.empty:
    values_to_append = pdf.values.tolist()
    ws.append_rows(values_to_append, value_input_option='USER_ENTERED')
    print(f"Successfully appended {len(values_to_append)} row(s) for Hour {pdf['Hour'].iloc[0]} to the 'Data' tab.")
else:
    print("No volume data found for the previous hour.")